https://www.cnblogs.com/tian777/p/15530142.html

# 对比

| 特性       | `logging` (标准库)                   | `loguru` (第三方库)        |
| -------- | --------------------------------- | ---------------------- |
| **设计哲学** | 灵活、可扩展、企业级                        | 简单、现代、零配置              |
| **学习曲线** | 陡峭（需要理解 Handler/Formatter/Filter） | 平缓（开箱即用）               |
| **配置方式** | 代码或配置文件（复杂）                       | 代码（简洁）                 |
| **线程安全** | ✅ 原生支持                            | ✅ 原生支持                 |
| **异步支持** | 需手动配置                             | ✅ 内置支持                 |
| **依赖**   | 无（内置）                             | 需 `pip install loguru` |


| 维度        | logging    | loguru           |
| --------- | ---------- | ---------------- |
| **代码量**   | 多（10+ 行配置） | 少（1 行导入）         |
| **可读性**   | 一般         | 优秀（彩色 + 简洁）      |
| **功能丰富度** | 基础（需扩展）    | 丰富（内置轮转/压缩/JSON） |
| **异常处理**  | 手动         | 自动（装饰器 + 堆栈美化）   |
| **性能**    | 良好         | 更好（异步内置）         |
| **生态兼容**  | 优秀（标准库）    | 良好（需适配部分库）       |


# 选择建议

| 场景         | 推荐          | 理由                     |
| ---------- | ----------- | ---------------------- |
| 快速原型/脚本    | **loguru**  | 零配置，立即使用               |
| 小型项目       | **loguru**  | 代码简洁，维护成本低             |
| 大型分布式系统    | **logging** | 生态成熟，与 ELK/Sentry 集成完善 |
| 需要严格合规     | **logging** | 标准库，审计友好               |
| 科学计算/AI 项目 | **loguru**  | 与 Jupyter、MLflow 配合好   |
| 嵌入式/受限环境   | **logging** | 无外部依赖                  |


<b>在实际生产环境中，直接使用 print() 是绝对禁止的，因为无法控制格式、无法分级、无法输出到文件、且并发下会乱码。</b>

标准的做法是使用 Python 内置的 logging 模块，并遵循以下最佳实践配置：
1. 模块化配置：在每个文件中通过 getLogger(__name__) 获取 logger。
2. 集中式配置：在主入口（__main__）统一配置 Handler（输出到哪里）、Formatter（输出格式）和 Level（日志级别）。
3. 结构化输出：包含时间、日志级别、文件名、行号、进程ID等信息。
4. 多Handler支持：同时输出到控制台（方便调试）和文件（方便归档和排查），甚至按天切割文件。


理解 logging 的关键在于理解它的五个核心组件及其协作关系：

1. <b>Logger (日志器)</b>
- 作用：应用程序直接打交道的对象。你通过它来记录日志（如 logger.info("消息")）。
- 特点：每个 Logger 都有一个名字（通常用 __name__），形成层级结构（如 a.b.c 是 a.b 的子节点）。
- 职责：判断日志是否达到设定级别（Level），决定是否传递给 Handler。
2. <b>Handler (处理器)</b>
- 作用：决定日志去哪里。
- 常见类型：
  - StreamHandler：输出到控制台（stdout/stderr）。
  - FileHandler：输出到文件。
  - RotatingFileHandler / TimedRotatingFileHandler：输出到文件，支持按大小或时间切割（防止文件无限变大）。
  - SMTPHandler：发送错误邮件。
- 职责：接收 Logger 传来的日志，根据自身的 Level 过滤，然后输出。
3. <b>Formatter (格式化器)</b>
- 作用：决定日志长什么样。
- 内容：定义时间、级别、文件名、行号、消息内容的排列格式。
- 绑定：只能绑定到 Handler 上（一个 Handler 只能有一个 Formatter）。

4. <b>Filter (过滤器)</b>
- 作用：更细粒度的控制（除了 Level 之外）。
- 场景：例如“只记录包含特定关键词的日志”或“排除某个模块的日志”。（进阶用法，初学者可暂略）。
5. <b>Level (日志级别)</b>
- 用于区分日志的重要程度。只有 >= 设定级别 的日志才会被处理。
- 默认级别（从低到高）：
| 级别 | 数值 | 场景描述 |
| :--- | :--- | :--- |
| `DEBUG` | 10 | 调试细节。变量值、函数入口出口。生产环境通常关闭。 |
| `INFO` | 20 | 一般信息。程序正常运行状态（如“启动成功”、“处理了100条数据”）。 |
| `WARNING` | 30 | 警告。发生了意料之外的小事，但程序还能跑（如“配置文件缺失，使用默认值”）。 |
| `ERROR` | 40 | 错误。功能失败，但程序未崩溃（如“数据库连接失败，重试中”）。 |
| `CRITICAL` | 50 | 严重错误。程序无法继续运行，即将崩溃。 |
| `NOTSET` | 0 | 不设置级别（通常继承父级）。 |

In [10]:
import logging

# 配置根 logger，级别为 INFO
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logging.debug("这条不会显示，因为级别低于 INFO")
logging.info("程序启动了")
logging.warning("注意：内存占用较高")
logging.error("出错了！")
logging.critical("系统崩溃！")

2026-03-03 17:33:27 | INFO     | 95624 | 1638053210.py:10 | <module> | 程序启动了
2026-03-03 17:33:27 | WARNING  | 95624 | 1638053210.py:11 | <module> | 注意：内存占用较高
2026-03-03 17:33:27 | ERROR    | 95624 | 1638053210.py:12 | <module> | 出错了！
2026-03-03 17:33:27 | CRITICAL | 95624 | 1638053210.py:13 | <module> | 系统崩溃！


| 方法 | 说明 | 示例 |
| :--- | :--- | :--- |
| `logger.debug(msg)` | 记录调试信息 | `logger.debug(f"Var x = {x}")` |
| `logger.info(msg)` | 记录一般信息 | `logger.info("User logged in")` |
| `logger.warning(msg)` | 记录警告 | `logger.warning("Config file missing")` |
| `logger.error(msg, exc_info=True)` | 记录错误。关键参数 `exc_info=True` 会自动附加异常堆栈跟踪。 | `logger.error("DB failed", exc_info=True)` |
| `logger.critical(msg)` | 记录严重错误 | `logger.critical("System down")` |
| `logger.exception(msg)` | 等价于 `logger.error(msg, exc_info=True)`。专门用在 `except` 块中。 | `except: logger.exception("Unknown error")` |
| `logger.log(level, msg)` | 自定义级别记录 | `logger.log(25, "Custom level")` |



logging 模块是 Python 可观测性的基石。
- 开发时：用 DEBUG 看清每一步。
- 测试时：用 INFO 确认流程。
- 生产时：用 WARNING/ERROR 监控异常，配合文件轮转和 exc_info 快速定位问题。

In [ ]:
这是一个模拟真实后端服务或数据处理任务的完整示例。它包含两个文件逻辑（主程序 + 子模块），展示了日志如何在模块间传递和记录。

my_project/
├── main.py            # 程序入口，负责配置 logging
├── worker.py          # 业务逻辑模块，使用 logger
└── logs/              # 自动创建的日志目录
    └── app.log

## worker.py 业务逻辑模块

注意：这里不需要配置handler，只需要获取logger

In [1]:
import logging
import time
import random

# 关键：使用__name__作为logger名称
# 这样日志会显示’worker‘而不是’root‘，便于区分来源

logger = logging.getLogger(__name__)

def process_data(data_id):
    logger.info(f'开始处理数据ID：{data_id}')
    
    try:
        # 模拟耗时操作
        time.sleep(0.5)
        
        # 模拟随机错误（20%概率失败）
        if random.random() < 0.2:
            raise ValueError(f'数据{data_id}格式校验失败')
        
        # 模拟警告情况
        if data_id %10 ==0:
            logger.warning(f'数据ID{data_id}处理较慢，建议优化')
        logger.debug(f'数据ID{data_id}中间计算结果：{data_id * 2}')
        logger.info(f'数据ID{data_id}处理成功')
        return True
    except ValueError as e:
        # 记录：警告级别的异常，不中断程序
        logger.warning(f'数据ID{data}')
        return False
    except Exception as e:
        # 记录错误级别的异常，包含完整堆栈信息
        # exc_info=True 是挂件，他会打印traceback
        logger.error(f'数据ID{data_id}发生未预期严重错误',exc_info=True)
        return False

def run_batch_task():
    logger.info('=== 开始批量任务 ===')
    success_count = 0
    for i in range(1,6):
        if process_data(i):
            success_count +=1
    
    logger.info(f'=== 批量任务结束，成功：{success_count}/5 ===')
    
        

## main.py 程序入口与日志配置

注意：所有复杂的配置都在这里完成


In [9]:
import logging
import os
import sys
from logging.handlers import RotatingFileHandler
# from worker import run_batch_task

def setup_logging():
    """
    配置日志系统：
    1. 控制台输出 (INFO 级别)
    2. 文件输出 (DEBUG 级别，支持轮转)
    """
    # 创建根 logger
    root_logger = logging.getLogger()
    root_logger.setLevel(logging.DEBUG)  # 根级别设低，让 handler 去过滤

    # 如果已经有 handler (避免在重复运行或测试时重复添加)，则清除
    if root_logger.hasHandlers():
        root_logger.handlers.clear()

    # --- 1. 定义格式器 (Formatter) ---
    # 格式：时间 | 级别 | 进程ID | 文件名:行号 | 函数名 | 消息
    log_format = logging.Formatter(
        fmt='%(asctime)s | %(levelname)-8s | %(process)d | %(filename)s:%(lineno)d | %(funcName)s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )

    # --- 2. 控制台 Handler (StreamHandler) ---
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)  # 控制台只显示 INFO 及以上
    console_handler.setFormatter(log_format)
    root_logger.addHandler(console_handler)

    # --- 3. 文件 Handler (RotatingFileHandler) ---
    # 确保日志目录存在
    log_dir = "logs"
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, "app.log")
    
    # 轮转策略：文件最大 5MB，保留 3 个备份文件 (app.log, app.log.1, app.log.2...)
    file_handler = RotatingFileHandler(
        log_file, 
        maxBytes=5 * 1024 * 1024,  # 5 MB
        backupCount=3,
        encoding='utf-8'
    )
    file_handler.setLevel(logging.DEBUG)  # 文件记录所有 DEBUG 及以上细节
    file_handler.setFormatter(log_format)
    root_logger.addHandler(file_handler)

    logging.info("日志系统初始化完成")

def main():
    # 1. 初始化日志
    setup_logging()
    
    logging.info("应用程序启动")
    
    try:
        # 2. 执行业务逻辑
        run_batch_task()
        
        # 模拟一个严重的崩溃场景用于演示 exc_info
        # logging.info("准备触发一个严重错误...")
        # x = 1 / 0 
        
    except KeyboardInterrupt:
        logging.warning("用户中断程序 (Ctrl+C)")
    except Exception as e:
        # 捕获主线程未处理的异常，防止程序静默死亡
        logging.critical("程序发生致命错误即将退出", exc_info=True)
        sys.exit(1)
    finally:
        logging.info("应用程序正常关闭")

# if __name__ == "__main__":
main()

2026-03-03 17:17:25 | INFO     | 95624 | 1916699627.py:51 | setup_logging | 日志系统初始化完成
2026-03-03 17:17:25 | INFO     | 95624 | 1916699627.py:57 | main | 应用程序启动
2026-03-03 17:17:25 | INFO     | 95624 | 2768590066.py:38 | run_batch_task | === 开始批量任务 ===
2026-03-03 17:17:25 | INFO     | 95624 | 2768590066.py:11 | process_data | 开始处理数据ID：1
2026-03-03 17:17:26 | INFO     | 95624 | 2768590066.py:25 | process_data | 数据ID1处理成功
2026-03-03 17:17:26 | INFO     | 95624 | 2768590066.py:11 | process_data | 开始处理数据ID：2
2026-03-03 17:17:26 | INFO     | 95624 | 2768590066.py:25 | process_data | 数据ID2处理成功
2026-03-03 17:17:26 | INFO     | 95624 | 2768590066.py:11 | process_data | 开始处理数据ID：3
2026-03-03 17:17:27 | INFO     | 95624 | 2768590066.py:25 | process_data | 数据ID3处理成功
2026-03-03 17:17:27 | INFO     | 95624 | 2768590066.py:11 | process_data | 开始处理数据ID：4
2026-03-03 17:17:27 | INFO     | 95624 | 2768590066.py:25 | process_data | 数据ID4处理成功
2026-03-03 17:17:27 | INFO     | 95624 | 2768590066.py:11 | proc

### exc_info=True:
- 这是生产环境排查问题的救命稻草。
- 如果不加这个参数，logger.error("出错了") 只会打印文字，不会打印堆栈跟踪（Traceback），你根本不知道错误发生在哪一行。
- 最佳实践：在 except Exception 块中记录 error 日志时，永远加上 exc_info=True

### RotatingFileHandler:
- 如果不使用轮转，日志文件会无限增长，最终撑爆磁盘。
- maxBytes 和 backupCount 确保磁盘占用可控，旧日志自动归档。

### 分离 Console 和 File 级别:
- Console (INFO): 开发运维人员看终端时，只关心主要流程，不想被大量的 DEBUG 信息刷屏。
- File (DEBUG): 当线上出 Bug 时，我们需要查看详细的变量状态和执行路径，这些详细信息只存文件，不干扰终端

### 根 Logger 配置:
- 我们在 main.py 配置了 root_logger。由于 worker.py 中的 logger 是 root 的子节点，它们会自动继承 root 的 handlers（除非显式设置 propagate=False）。这使得我们只需要在入口文件配置一次，整个项目的所有模块都能自动输出日志。

### 如何在 Docker/K8s 中使用？
在容器化环境中，最佳实践通常是只输出到 stdout (控制台)，由 Docker 或 K8s 的日志收集代理（如 Fluentd, Filebeat）去采集文件或直接采集标准输出。

修改 setup_logging 即可适配：

In [ ]:
def setup_logging_for_docker():
    root_logger = logging.getLogger()
    root_logger.setLevel(logging.INFO)
    
    # 只保留控制台 Handler，格式调整为 JSON (方便机器解析)
    # 需要安装 python-json-logger 库
    from pythonjsonlogger import jsonlogger
    
    handler = logging.StreamHandler(sys.stdout)
    formatter = jsonlogger.JsonFormatter(
        fmt="%(asctime)s %(name)s %(levelname)s %(message)s %(filename)s %(lineno)d"
    )
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)